# CineData Analytics | Bronze → Silver

**Objetivo:** limpar, tipar, padronizar e deduplicar os dados da Bronze.

**Princípios aplicados em todas as tabelas:**
- A Bronze **nunca** é alterada: apenas lida.
- Colunas renomeadas para **português** e com **tipagem correta**.
- Conversões usam `try_cast` / `try_to_timestamp`: valor incompatível vira `NULL` **sem derrubar o pipeline**
  (importante por causa do *Column Shift* da base bruta).
- Tabelas Silver são reescritas por completo a cada execução (`overwrite`), o que torna o notebook idempotente.

In [ ]:
from functools import reduce

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

In [ ]:
dbutils.widgets.text("catalogo", "workspace", "Catálogo")
CATALOGO = dbutils.widgets.get("catalogo").strip()

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Camada Silver - dados limpos, tipados e deduplicados'")

# Parser de datas do Spark 3+ (sem ambiguidade com o parser legado)
try:
    spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")
except Exception:  # noqa: BLE001 - alguns ambientes serverless não permitem alterar a conf
    pass

## Funções utilitárias (reaproveitadas pelas 7 tabelas)

In [ ]:
# Textos que, na origem, representam AUSÊNCIA de dado (comparação feita em maiúsculas)
TOKENS_AUSENCIA = [
    "", "UNKNOWN", "NÃO INFORMADO", "NAO INFORMADO", "N/A", "NA", "N.A.", "NULL", "NONE",
    "NAN", "-", "--", "?", "SEM INFORMAÇÃO", "SEM INFORMACAO", "NOT AVAILABLE", "TBD", "NENHUM", "[]",
]


def texto_limpo(coluna: str):
    """trim + converte tokens de ausência ('Unknown', 'Não Informado', '', ...) em NULL."""
    c = F.trim(F.col(coluna))
    return F.when(F.upper(c).isin(TOKENS_AUSENCIA), F.lit(None).cast("string")).otherwise(c)


def limpar_id(coluna: str = "id"):
    """Chave natural do filme: trim e remoção de sufixo '.0' (id lido como float em alguma extração)."""
    return F.regexp_replace(texto_limpo(coluna), r"\.0+$", "")


def try_cast(coluna: str, tipo: str):
    """Conversão segura: devolve NULL em vez de erro quando o valor é incompatível (ex.: texto em coluna numérica)."""
    return F.expr(f"try_cast(`{coluna}` AS {tipo})")


def normalizar_numero(df: DataFrame, origem: str, destino: str, milhar_grupo_unico: bool = True,
                      remover_simbolos: bool = True) -> DataFrame:
    """
    Higieniza um número armazenado como texto e devolve uma STRING no padrão '1234.56', pronta para try_cast.

    1) tokens de ausência -> NULL; 2) remove tudo que não é dígito, ',', '.' ou '-' (símbolos de moeda, 'USD', espaços);
    3) decide qual separador é decimal e qual é de milhar:
       - '1.234.567,89' -> 1234567.89   (padrão BR)       - '1,234,567.89' -> 1234567.89 (padrão US)
       - '1.234.567'    -> 1234567      (vários grupos de 3 = milhar)
       - '12,5'         -> 12.5         (vírgula decimal)
       - '1.500' / '1,500': grupo único de 3 dígitos é ambíguo. Para dinheiro/contagens (milhar_grupo_unico=True)
         tratamos como milhar; para popularidade/notas (False) tratamos como decimal (ex.: popularidade 12.345).
    remover_simbolos=False (métricas): se houver QUALQUER caractere além de dígito, sinal e separador, o valor é
    resíduo de Column Shift (ex.: '214"', 'English', trecho de sinopse) e vira NULL em vez de ser "consertado".
    """
    if remover_simbolos:
        df = df.withColumn("_n", F.regexp_replace(texto_limpo(origem), r"[^0-9,.\-]", ""))
    else:
        t = texto_limpo(origem)
        df = df.withColumn("_n", F.when(t.rlike(r"^-?[0-9]+([.,][0-9]+)*$"), t))
    n = F.col("_n")
    tem_ponto, tem_virgula = n.contains("."), n.contains(",")

    expr = (
        F.when(n.isNull() | n.isin("", "-", ".", ","), F.lit(None).cast("string"))
        .when(tem_ponto & tem_virgula & n.rlike(r",[0-9]{1,2}$"),
              F.regexp_replace(F.regexp_replace(n, r"\.", ""), ",", "."))
        .when(tem_ponto & tem_virgula, F.regexp_replace(n, ",", ""))
        .when(n.rlike(r"^-?[0-9]{1,3}(,[0-9]{3}){2,}$"), F.regexp_replace(n, ",", ""))
        .when(n.rlike(r"^-?[0-9]{1,3}(\.[0-9]{3}){2,}$"), F.regexp_replace(n, r"\.", ""))
    )
    if milhar_grupo_unico:
        expr = (expr.when(n.rlike(r"^-?[0-9]{1,3},[0-9]{3}$"), F.regexp_replace(n, ",", ""))
                    .when(n.rlike(r"^-?[0-9]{1,3}\.[0-9]{3}$"), F.regexp_replace(n, r"\.", "")))
    expr = expr.when(tem_virgula, F.regexp_replace(n, ",", ".")).otherwise(n)
    return df.withColumn(destino, expr).drop("_n")


def manter_mais_recente(df: DataFrame, chave: str = "id_filme") -> DataFrame:
    """
    Garante 1 linha por chave mantendo a versão mais recente (maior ingestion_datetime).
    Empate (duplicata dentro do mesmo lote): fica o registro mais completo (mais colunas não nulas).
    """
    colunas_negocio = [c for c in df.columns if c not in (chave, "ingestion_datetime")]
    completude = reduce(lambda a, b: a + b, [F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in colunas_negocio])
    janela = Window.partitionBy(chave).orderBy(F.col("ingestion_datetime").desc(), completude.desc())
    return (df.withColumn("_rn", F.row_number().over(janela))
              .filter("_rn = 1")
              .drop("_rn", "ingestion_datetime"))


def salvar(df: DataFrame, tabela: str) -> None:
    """Overwrite completo: a Silver é sempre reconstruída a partir da Bronze (idempotente)."""
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    print(f"{tabela:<35} | {spark.table(tabela).count():>8} linhas")

## 7) `silver.tb_cotacao_dolar` — processada primeiro, pois a tabela financeira depende dela
A API não publica cotação em fins de semana e feriados. Montamos um **calendário contínuo** (do primeiro dia com
cotação até hoje) e aplicamos **Forward Fill**: o dia sem cotação recebe o valor do último dia útil disponível.

In [ ]:
cot_bruta = (
    spark.table("bronze.tb_cotacao_dolar")
    .withColumn("data_hora_cotacao", try_cast("dataHoraCotacao", "TIMESTAMP"))
    .withColumn("cotacao_compra", try_cast("cotacaoCompra", "DECIMAL(10,4)"))
    .filter(F.col("data_hora_cotacao").isNotNull() & (F.col("cotacao_compra") > 0))
    .withColumn("data_cotacao", F.to_date("data_hora_cotacao"))
)

# Um valor por dia: o boletim mais tardio do dia (e o lote mais recente, caso a mesma data venha em várias cargas)
janela_dia = Window.partitionBy("data_cotacao").orderBy(F.col("data_hora_cotacao").desc(),
                                                       F.col("ingestion_datetime").desc())
cot_diaria = (cot_bruta.withColumn("_rn", F.row_number().over(janela_dia))
                       .filter("_rn = 1")
                       .select("data_cotacao", "cotacao_compra"))

calendario = (
    cot_diaria.agg(F.min("data_cotacao").alias("ini"),
                   F.greatest(F.max("data_cotacao"), F.current_date()).alias("fim"))
    .select(F.explode(F.sequence("ini", "fim", F.expr("INTERVAL 1 DAY"))).alias("data_cotacao"))
)

# Forward fill: último valor não nulo até a linha atual (série pequena, janela global sem partição é aceitável)
janela_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)
silver_cotacao = (
    calendario.join(cot_diaria, "data_cotacao", "left")
    .withColumn("cotacao_preenchida", F.col("cotacao_compra").isNull())   # True = dia sem cotação oficial
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(janela_ffill))
    .select("data_cotacao", "cotacao_compra", "cotacao_preenchida")
    .orderBy("data_cotacao")
)
salvar(silver_cotacao, "silver.tb_cotacao_dolar")
display(spark.table("silver.tb_cotacao_dolar").orderBy(F.desc("data_cotacao")).limit(10))

In [ ]:
# Taxa usada na conversão para BRL: cotação (compra) do dia mais recente da série contínua
ultima = spark.table("silver.tb_cotacao_dolar").orderBy(F.desc("data_cotacao")).first()
TAXA_DOLAR, DATA_TAXA = ultima["cotacao_compra"], ultima["data_cotacao"]
if TAXA_DOLAR is None:
    raise RuntimeError("Não há cotação disponível para converter os valores para BRL.")
print(f"Taxa aplicada: R$ {TAXA_DOLAR} (referência {DATA_TAXA})")

## 1) `silver.tb_info_filmes`
- **Status:** normalização antes da tradução. Removemos tudo que não é letra (ruídos, hífens, espaços) e
  colocamos em minúsculas: `'--POST-Production '` → `postproduction`. Só então traduzimos pelo dicionário;
  o que não casa (registro corrompido) vira **'Não Informado'**.
- **Data multi-formato:** testamos vários padrões com `try_to_timestamp` e ficamos com o primeiro que funcionar
  (`coalesce`). Só vira NULL o que nenhum padrão consegue ler. Perfil da origem: ~91 mil `yyyy-MM-dd`,
  ~10,5 mil `dd/MM/yyyy` e ~4,7 mil `MM-dd-yyyy`. A convenção de cada separador foi confirmada nos próprios dados:
  com barra, 6,2 mil registros têm o 1º número > 12 (logo é **dia**/mês); com hífen, 2,7 mil têm o 2º número > 12
  (logo é **mês**-dia). Nenhum registro contradiz isso, então as datas ambíguas seguem a mesma convenção.
- **Deduplicação:** 1 linha por filme, mantendo o registro do lote de ingestão mais recente.

In [ ]:
MAPA_STATUS = {
    "released": "Lançado",
    "postproduction": "Pós-Produção",
    "inproduction": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "rumoured": "Rumores",
    "canceled": "Cancelado",
    "cancelled": "Cancelado",
}
mapa_status = F.create_map(*[F.lit(x) for par in MAPA_STATUS.items() for x in par])
status_normalizado = F.regexp_replace(F.lower(F.col("status")), r"[^a-z]", "")

# Ordem importa: ISO; barra = dia/mês/ano; hífen = mês-dia-ano (convenções verificadas na origem); extensos por último
FORMATOS_DATA = [
    "yyyy-M-d", "yyyy/M/d", "yyyy.M.d", "yyyyMMdd",
    "d/M/yyyy", "M-d-yyyy", "d.M.yyyy",
    "MMM d, yyyy", "MMMM d, yyyy", "MMM d yyyy", "MMMM d yyyy",
    "d MMM yyyy", "d MMMM yyyy", "d-MMM-yyyy",
]
# Remove parte de hora (ex.: '2019-05-01 00:00:00' ou '2019-05-01T00:00:00Z') antes de testar os formatos
data_como_texto = F.regexp_replace(texto_limpo("release_date"), r"[T ][0-9]{1,2}:[0-9]{2}.*$", "")
data_convertida = F.coalesce(*[F.expr(f"to_date(try_to_timestamp(_data_txt, '{fmt}'))") for fmt in FORMATOS_DATA])

info = (
    spark.table("bronze.tb_movies_info")
    .withColumn("_data_txt", data_como_texto)
    .select(
        limpar_id("id").alias("id_filme"),
        texto_limpo("title").alias("titulo"),
        texto_limpo("original_title").alias("titulo_original"),
        data_convertida.alias("data_lancamento"),
        # '120 min' / '120.0' -> 120; zero, negativo ou absurdo (> 1000 min) não é duração válida
        try_cast("runtime", "DOUBLE").alias("_duracao_num"),
        F.regexp_extract(F.col("runtime"), r"([0-9]+)", 1).alias("_duracao_txt"),
        F.lower(texto_limpo("original_language")).alias("idioma_original"),
        F.coalesce(mapa_status[status_normalizado], F.lit("Não Informado")).alias("status_filme"),
        texto_limpo("overview").alias("sinopse"),
        texto_limpo("tagline").alias("frase_divulgacao"),
        "ingestion_datetime",
    )
    .withColumn("duracao_minutos",
                F.coalesce(F.col("_duracao_num").cast("int"), try_cast("_duracao_txt", "INT")))
    .withColumn("duracao_minutos",
                F.when(F.col("duracao_minutos").between(1, 1000), F.col("duracao_minutos")))
    .drop("_duracao_num", "_duracao_txt")
    # Sanidade: anos fora do intervalo plausível do cinema vêm de parse errado de lixo numérico
    .withColumn("data_lancamento",
                F.when(F.year("data_lancamento").between(1874, 2100), F.col("data_lancamento")))
    .filter(F.col("id_filme").isNotNull())
)

silver_info = (
    manter_mais_recente(info, "id_filme")
    .withColumn("ano_lancamento", F.year("data_lancamento"))   # coluna derivada exigida
    .select("id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento", "duracao_minutos",
            "idioma_original", "status_filme", "sinopse", "frase_divulgacao")
)
salvar(silver_info, "silver.tb_info_filmes")

In [ ]:
# Conferências: distribuição do status traduzido e quantas datas não puderam ser convertidas
display(spark.table("silver.tb_info_filmes").groupBy("status_filme").count().orderBy(F.desc("count")))
nao_convertidas = (spark.table("bronze.tb_movies_info").select(limpar_id("id").alias("id_filme"), "release_date")
                   .join(spark.table("silver.tb_info_filmes").filter("data_lancamento IS NULL"), "id_filme")
                   .filter(texto_limpo("release_date").isNotNull()))
print(f"Filmes com data preenchida na origem mas impossível de converter: {nao_convertidas.select('id_filme').distinct().count()}")
display(nao_convertidas.select("release_date").distinct().limit(20))

## 2) `silver.tb_financeiro_filmes`
- Tokens de ausência (`Unknown`, `Não Informado`...) → NULL **antes** da conversão.
- Remoção de símbolos de moeda (`$`, `USD`) e separadores de milhar (`normalizar_numero`) → `DECIMAL(18,2)`.
  Valores abreviados com sufixo de escala (`34.0M`, `150.0K`) são expandidos (× 1 milhão / × mil).
- Orçamento/receita **zerados ou negativos** não são valores reais: viram NULL.
- BRL = USD × cotação mais recente da série contínua.
- **Lucro** = receita − orçamento. Decisão de negócio: se um dos dois estiver ausente o lucro fica **NULL**
  (tratar o ausente como 0 inventaria lucro/prejuízo). A tabela não é invalidada: o filme continua com os
  demais campos. **Margem %** = lucro / receita × 100, calculada só quando receita > 0 (sem divisão por zero).

In [ ]:
def valor_monetario_usd(df: DataFrame, origem: str, destino: str) -> DataFrame:
    """
    Converte o texto monetário em DECIMAL(18,2). Formatos vistos na origem: '58000000', '$ 97000000',
    'USD 150000000', '34.0M' (milhões), '150.0K' (milhares), 'N/A', 'Unknown', 'Não Informado'.
    Valores com sufixo de escala (K/M/B) são multiplicados; os demais passam pela higienização padrão.
    """
    txt = F.upper(F.trim(F.col(origem)))
    escala = (F.when(txt.rlike(r"[0-9]\s*K$"), F.lit(1e3))
               .when(txt.rlike(r"[0-9]\s*M$"), F.lit(1e6))
               .when(txt.rlike(r"[0-9]\s*B$"), F.lit(1e9)))
    df = normalizar_numero(df, origem, "_normalizado", milhar_grupo_unico=True)
    numero_com_escala = F.expr(f"try_cast(regexp_extract(`{origem}`, '([0-9]+([.][0-9]+)?)', 1) AS DOUBLE)") * escala
    return (df.withColumn(destino, F.when(escala.isNotNull(), numero_com_escala.cast("DECIMAL(18,2)"))
                                     .otherwise(try_cast("_normalizado", "DECIMAL(18,2)")))
              .drop("_normalizado"))


fin = spark.table("bronze.tb_movies_financials").withColumn("id_filme", limpar_id("id"))
fin = valor_monetario_usd(fin, "budget", "orcamento_usd")
fin = valor_monetario_usd(fin, "revenue", "receita_usd")

fin = (
    fin.select("id_filme", "ingestion_datetime", "orcamento_usd", "receita_usd")
    .withColumn("orcamento_usd", F.when(F.col("orcamento_usd") > 0, F.col("orcamento_usd")))
    .withColumn("receita_usd", F.when(F.col("receita_usd") > 0, F.col("receita_usd")))
    .filter(F.col("id_filme").isNotNull())
)

taxa = F.lit(TAXA_DOLAR).cast("DECIMAL(10,4)")
silver_fin = (
    manter_mais_recente(fin, "id_filme")
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * taxa).cast("DECIMAL(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * taxa).cast("DECIMAL(18,2)"))
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("DECIMAL(18,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("DECIMAL(18,2)"))
    .withColumn("margem_lucro_pct",
                F.when(F.col("receita_usd") > 0,
                       F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2).cast("DECIMAL(10,2)")))
    .withColumn("taxa_cambio", taxa)
    .withColumn("data_cotacao", F.lit(DATA_TAXA).cast("date"))
)
salvar(silver_fin, "silver.tb_financeiro_filmes")
display(spark.table("silver.tb_financeiro_filmes").limit(10))

## 3) `silver.tb_metricas_engajamento`
- **Popularidade:** limpeza de separadores antes da conversão (`12,345` e `12.345` = 12.345; `1.234.567,8` = 1234567.8),
  evitando que o valor vire NULL silenciosamente.
- **Column Shift:** conversão segura; texto em coluna numérica vira NULL sem quebrar a execução.
- **Regras de negócio:** notas fora de 0–10 (inclusive as multiplicadas por erro de escala, ex.: 75 em vez de 7.5)
  → NULL. Contagem de votos e popularidade negativas → NULL. Contagem de votos precisa ser inteira.

In [ ]:
met = spark.table("bronze.tb_movies_metrics").withColumn("id_filme", limpar_id("id"))
met = normalizar_numero(met, "popularity", "_pop", milhar_grupo_unico=False, remover_simbolos=False)
met = normalizar_numero(met, "vote_average", "_nota_tmdb", milhar_grupo_unico=False, remover_simbolos=False)
met = normalizar_numero(met, "averageRating", "_nota_imdb", milhar_grupo_unico=False, remover_simbolos=False)
# Contagens de voto na origem nunca usam separador de milhar; um '6.891' na coluna de votos é a nota média
# deslocada pelo Column Shift. Por isso grupo único NÃO é tratado como milhar: vira 6.891 -> não inteiro -> NULL.
met = normalizar_numero(met, "vote_count", "_votos_tmdb", milhar_grupo_unico=False, remover_simbolos=False)
met = normalizar_numero(met, "numVotes", "_votos_imdb", milhar_grupo_unico=False, remover_simbolos=False)


def nota_valida(coluna: str):
    v = try_cast(coluna, "DOUBLE")
    return F.when(v.between(0, 10), v)


def contagem_valida(coluna: str):
    # Contagem precisa ser inteira e não negativa; '7.5' numa coluna de votos é resíduo de Column Shift
    v = try_cast(coluna, "DOUBLE")
    return F.when((v >= 0) & (v == F.floor(v)) & (v <= 2147483647), v.cast("int"))


pop = try_cast("_pop", "DOUBLE")
met = met.select(
    "id_filme", "ingestion_datetime",
    F.when(pop >= 0, pop).alias("popularidade"),
    nota_valida("_nota_tmdb").alias("nota_media_tmdb"),
    contagem_valida("_votos_tmdb").alias("qtd_votos_tmdb"),
    nota_valida("_nota_imdb").alias("nota_media_imdb"),
    contagem_valida("_votos_imdb").alias("qtd_votos_imdb"),
).filter(F.col("id_filme").isNotNull())

salvar(manter_mais_recente(met, "id_filme"), "silver.tb_metricas_engajamento")
display(spark.table("silver.tb_metricas_engajamento").summary("count", "min", "max"))

## 4) `silver.tb_avaliacoes_usuarios`
- Nota do usuário fora da escala 0–10 → NULL.
- Comentário nulo, vazio ou só com espaços → **'Sem comentário'**.
- Remoção de duplicatas integrais (filme + usuário + nota + comentário), feita **depois** da padronização,
  para que `'  ótimo '` e `'ótimo'` sejam reconhecidos como a mesma avaliação. Como não usamos o
  `ingestion_datetime` na comparação, cargas repetidas da Bronze também são eliminadas.

In [ ]:
rev = spark.table("bronze.tb_movies_reviews")
rev = normalizar_numero(rev, "nota", "_nota", milhar_grupo_unico=False, remover_simbolos=False)
comentario = F.regexp_replace(F.trim(F.col("comentario")), r"\s+", " ")

silver_rev = (
    rev.select(
        limpar_id("id").alias("id_filme"),
        texto_limpo("nome").alias("nome_usuario"),
        nota_valida("_nota").alias("nota_usuario"),
        F.when(comentario.isNull() | (comentario == ""), F.lit("Sem comentário"))
         .otherwise(comentario).alias("comentario_usuario"),
    )
    .filter(F.col("id_filme").isNotNull())
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
)
salvar(silver_rev, "silver.tb_avaliacoes_usuarios")

## 5) `silver.tb_generos` e 6) `silver.tb_pessoas_empresas`
A coluna de origem guarda vários valores numa só célula, com separadores inconsistentes (`,` `;` `|`).
Unificamos o separador, fazemos `split` + `posexplode` (a posição preserva a **ordem de crédito**, útil para
achar os atores principais na Gold) e removemos resíduos do *Column Shift*.

**Gêneros:** validação por **domínio** — só entram gêneros do catálogo TMDB (com sinônimos/variações de escrita
e nomes em português mapeados para o nome canônico). Números, frases descritivas e nomes de pessoas deslocados
para a coluna são descartados (a célula de conferência mostra o que foi descartado).

In [ ]:
cred = spark.table("bronze.tb_credits_and_tags").withColumn("id_filme", limpar_id("id"))
cred = manter_mais_recente(
    cred.select("id_filme", "genres", "cast", "directors", "writers", "production_companies", "ingestion_datetime")
        .filter(F.col("id_filme").isNotNull()),
    "id_filme",
)


def explodir(coluna: str, separadores: str = r"[,;|]"):
    """split + posexplode tratando separadores inconsistentes. Retorna id_filme, ordem, valor_bruto."""
    return (cred.select("id_filme", F.posexplode(F.split(F.coalesce(F.col(coluna), F.lit("")), separadores))
                        .alias("ordem", "valor_bruto")))


def chave_texto(c):
    """Chave de comparação: minúsculas, sem acento, só letras."""
    sem_acento = F.translate(F.lower(F.trim(c)), "áàâãäéèêëíìîïóòôõöúùûüç", "aaaaaeeeeiiiiooooouuuuc")
    return F.regexp_replace(sem_acento, r"[^a-z]", "")


GENEROS_CANONICOS = {
    "Action": ["action", "acao"], "Adventure": ["adventure", "aventura"],
    "Animation": ["animation", "animacao"], "Comedy": ["comedy", "comedia"], "Crime": ["crime"],
    "Documentary": ["documentary", "documentario"], "Drama": ["drama"],
    "Family": ["family", "familia"], "Fantasy": ["fantasy", "fantasia"], "History": ["history", "historia"],
    "Horror": ["horror", "terror"], "Music": ["music", "musica"], "Mystery": ["mystery", "misterio"],
    "Romance": ["romance"], "Science Fiction": ["sciencefiction", "scifi", "ficcaocientifica"],
    "TV Movie": ["tvmovie", "cinematv", "filmedetv"], "Thriller": ["thriller", "suspense"],
    "War": ["war", "guerra"], "Western": ["western", "faroeste"],
}
mapa_generos = F.create_map(*[F.lit(x) for canonico, chaves in GENEROS_CANONICOS.items()
                              for chave in chaves for x in (chave, canonico)])

generos_explodidos = explodir("genres", r"[,;|/]").withColumn("nome_genero", mapa_generos[chave_texto(F.col("valor_bruto"))])

silver_generos = (
    generos_explodidos.filter(F.col("nome_genero").isNotNull())
    .select("id_filme", "nome_genero")
    .dropDuplicates()
)
salvar(silver_generos, "silver.tb_generos")

print("Valores descartados da coluna genres (resíduos de Column Shift / fora do domínio):")
display(generos_explodidos.filter(F.col("nome_genero").isNull() & (F.trim("valor_bruto") != ""))
        .groupBy("valor_bruto").count().orderBy(F.desc("count")).limit(30))

**Pessoas e empresas:** dimensão unificada `cast → Ator`, `directors → Diretor`, `writers → Roteirista`,
`production_companies → Produtora`. Limpeza: remove aspas/colchetes das pontas, espaços duplicados, valores
numéricos, tokens de ausência, textos descritivos longos (frases deslocadas) e nomes de gênero que caíram na
coluna errada. Capitalização padronizada com `initcap` e deduplicação por filme + nome + tipo (mantendo a
menor posição de crédito).

In [ ]:
TIPOS_ENTIDADE = {"cast": "Ator", "directors": "Diretor", "writers": "Roteirista", "production_companies": "Produtora"}
chaves_genero = [chave for chaves in GENEROS_CANONICOS.values() for chave in chaves]

entidades = reduce(DataFrame.unionByName, [
    explodir(coluna).withColumn("tipo_entidade", F.lit(tipo)) for coluna, tipo in TIPOS_ENTIDADE.items()
])

nome = F.regexp_replace(F.col("valor_bruto"), r"^[\s\"'\[\]()]+|[\s\"'\[\]()]+$", "")
nome = F.regexp_replace(nome, r"\s+", " ")
qtd_palavras = F.size(F.split(nome, " "))

silver_pessoas = (
    entidades.withColumn("nome_limpo", nome)
    .filter(
        (F.length("nome_limpo") >= 2)
        & (F.length("nome_limpo") <= 80)
        & (qtd_palavras <= 8)                                       # frases descritivas deslocadas
        & F.col("nome_limpo").rlike(r"[A-Za-zÀ-ÿ]")                  # precisa ter letra (descarta números)
        & ~F.upper(F.col("nome_limpo")).isin(TOKENS_AUSENCIA)
        & ~chave_texto(F.col("nome_limpo")).isin(chaves_genero)     # gênero caído na coluna errada
        & ~F.col("nome_limpo").rlike(r"(?i)^/|https?:|\.(jpg|jpeg|png)$")  # caminhos de imagem/URLs deslocados
    )
    .withColumn("nome_entidade", F.initcap(F.col("nome_limpo")))
    .groupBy("id_filme", "nome_entidade", "tipo_entidade")
    .agg(F.min("ordem").alias("ordem_credito"))
)
salvar(silver_pessoas, "silver.tb_pessoas_empresas")
display(spark.table("silver.tb_pessoas_empresas").groupBy("tipo_entidade").count())

## Resumo da camada Silver

In [ ]:
for t in ["tb_info_filmes", "tb_financeiro_filmes", "tb_metricas_engajamento", "tb_avaliacoes_usuarios",
          "tb_generos", "tb_pessoas_empresas", "tb_cotacao_dolar"]:
    print(f"silver.{t:<28} {spark.table(f'silver.{t}').count():>8} linhas")